## Aula 7 – Pipeline de KDT

Este notebook demonstra, de forma simples, completa e reprodutível (sem downloads externos), um pipeline típico de
Descoberta de Conhecimento em Textos (KDT).

### Etapas do pipeline
1. Corpus (dados brutos)
2. Pré-processamento
3. Representação do texto
4. Descoberta de estruturas latentes (tópicos)
5. Organização do corpus (clustering)
6. Estruturação de conhecimento

O KDT não está em uma única célula do notebook.
Ele está no encadeamento das etapas e, principalmente, nas saídas estruturadas finais.

Ou seja:
👉 KDT é o processo, não um algoritmo específico.

## 1. Seleção do corpus – Dados Textuais Brutos

Criaremos um corpus simples com 3 temas, compostos por 10 documentos cada:

- **Espaço** (astronomia, satélites, foguetes)
- **Esportes** (futebol)
- **Computação gráfica** (renderização, GPU, imagem)

Em um projeto real, esta etapa viria de:
- arquivos (PDF, TXT, CSV)
- bancos de dados
- scraping (quando permitido)

In [3]:
space_docs = [
    "A NASA lançou um foguete para colocar um satélite em órbita e estudar a atmosfera.",
    "Astrônomos analisaram dados de um telescópio espacial para observar galáxias distantes.",
    "A missão a Marte depende de propulsão eficiente, comunicação por rádio e planejamento orbital.",
    "O satélite enviou imagens da Terra e medições de radiação no cinturão de Van Allen.",
    "A agência espacial discutiu janelas de lançamento, trajetória e reentrada controlada.",
    "Cientistas monitoraram a atividade solar e seus efeitos em comunicações e navegação.",
    "O rover coletou amostras de solo marciano e transmitiu resultados para a base.",
    "O telescópio detectou exoplanetas pela variação de brilho e espectroscopia.",
    "O foguete reutilizável reduz custos e amplia a frequência de missões em órbita baixa.",
    "A estação espacial realizou experimentos de microgravidade com novos materiais."
]

football_docs = [
    "O time marcou gol após jogada pelo meio campo e venceu a partida.",
    "O técnico ajustou o esquema tático e reforçou o meio campo.",
    "O atacante marcou gol e decidiu o jogo no segundo tempo.",
    "A defesa falhou na marcação e sofreu gol em contra ataque.",
    "O goleiro fez defesa importante após chute dentro da área.",
    "O clássico teve rivalidade, torcida e muitos gols.",
    "O árbitro marcou falta e aplicou cartão durante a partida.",
    "A equipe dominou o meio campo e criou chances de gol.",
    "O campeonato segue equilibrado com disputa por pontos e gols.",
    "O atacante perdeu pênalti mas marcou gol depois."
]

graphics_docs = [
    "A GPU acelera a renderização 3D com shaders e texturas no pipeline gráfico.",
    "O pipeline de renderização usa GPU, shader e textura para desenhar a cena em tempo real.",
    "Shaders controlam iluminação e materiais, enquanto a GPU aplica texturas na renderização.",
    "A renderização em tempo real depende de GPU e do pipeline gráfico com shaders otimizados.",
    "Ray tracing melhora sombras e reflexos, mas exige muita GPU e ajustes de renderização.",
    "A GPU processa shaders de pixel e vertex, aplicando texturas na renderização 3D.",
    "O motor gráfico organiza o pipeline, envia texturas e executa shaders na GPU.",
    "Para manter a taxa de quadros, o pipeline reduz custo de shader e otimiza texturas na GPU.",
    "A renderização 3D combina GPU, shader, textura e pipeline para produzir imagens realistas.",
    "O sistema ajustou o pipeline de renderização e reduziu o custo de shaders na GPU."
]

documents = space_docs + football_docs + graphics_docs

labels = (
    ["espaco"] * len(space_docs) +
    ["futebol"] * len(football_docs) +
    ["computacao_grafica"] * len(graphics_docs)
)

print("Total de documentos:", len(documents))
print("Exemplo:", documents[0]) # mostra apenas o 1o documento


Total de documentos: 30
Exemplo: A NASA lançou um foguete para colocar um satélite em órbita e estudar a atmosfera.


## 2. Pré-processamento (limpeza + normalização)

Nesta etapa, realizamos a limpeza e padronização dos dados para reduzir o ruído:  
- Conversão para minúsculas.
- Remoção de URLs e caracteres especiais (preservando a acentuação básica).
- Normalização de espaços em branco.

> Em casos mais avançados, incluiríamos lematização e tratamento mais robusto de Unicode.


In [4]:
import re

def simple_preprocess_pt(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # mantém letras (inclui acentos comuns), espaço
    text = re.sub(r"[^a-záàâãéêíóôõúçñ\s]", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

documents_clean = [simple_preprocess_pt(t) for t in documents]

print("Depois:", documents_clean[0:2]) # mostra apenas 2 documentos

Depois: ['a nasa lançou um foguete para colocar um satélite em órbita e estudar a atmosfera', 'astrônomos analisaram dados de um telescópio espacial para observar galáxias distantes']


## 3. Representação do texto (TF-IDF)

Após esse passo:
- Não há mais palavras
- Apenas vetores numéricos
- Todo o restante (tópicos, clustering, grafos) opera sobre números.

O método `fit_transform` executa duas etapas distintas:
- fit → aprende o modelo a partir do corpus. O `TfidfVectorizer` analisa todo o corpus e aprende quais termos aparecem nos documentos, quais termos são descartados observando-se as `stop_words` e as frequências mínima e máxima;
- transform → aplica esse modelo já treinado ao corpus, gerando a matriz TF-IDF.

Ou seja: `fit_transform = aprender + aplicar`

No **fit**, o modelo calcula o IDF (Inverse Document Frequency): ajuda a penalizar termos excessivamente comuns no corpus e valorizar termos que trazem maior distinção semântica:
$$\text{IDF}(t)=\log\frac{N}{df(t)}$$

Onde:
- $N$ = número total de documentos,
- $df(t)$ = número de documentos que contêm o termo
$t$.

➡️ Termos muito frequentes recebem menor peso  
➡️ Termos mais raros recebem maior peso

Esse passo exige o corpus inteiro, por isso ele acontece no fit.

No **transform**, o modelo já treinado é aplicado documento a documento:
- Cálculo do TF (Term Frequency)  
  Para cada documento:
  - Conta quantas vezes cada termo do vocabulário aparece
  - Normaliza essa frequência  
- Combinação TF × IDF  
  Para cada documento e termo:
  $$\text{TF}-\text{IDF}(d,t)=\text{TF}(d,t)\times\text{IDF}(t)$$

O resultado é uma matriz numérica:
- Linhas → documentos
- Colunas → termos do vocabulário
- Valores → pesos TF-IDF

> Transformamos documentos pré-processados(`documents_clean`) em uma matriz de pesos.
Entrada: `documents_clean` → Saída: `X_tfidf`


In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

stopwords_pt = ["a","o","os","as","um","uma","de","do","da","e","em","para","com","no","na"]

vectorizer = TfidfVectorizer(stop_words=stopwords_pt)
X_tfidf = vectorizer.fit_transform(documents_clean)
vocab = np.array(vectorizer.get_feature_names_out())

print("Matriz TF-IDF:", X_tfidf.shape)

Matriz TF-IDF: (30, 176)


### Imprimindo os termos mais significativos por documento (Top-N TF-IDF)

Este código percorre cada documento da matriz `X_tfidf` e seleciona apenas os termos com maior peso TF-IDF naquele documento.

Para cada documento:
- O vetor TF-IDF é convertido em um array denso;
- Os termos são ordenados por peso TF-IDF em ordem decrescente;
- Apenas os N termos mais significativos (definidos por `TOP_N`) são exibidos;
- Cada termo é apresentado juntamente com:
  - seu valor TF-IDF;
  - a forma textual correspondente no vocabulário (`vocab`);
  - o texto original do documento (`texts`)


In [6]:
import numpy as np

TOP_N = 8  # número de termos mais significantes por documento

for doc_idx in range(X_tfidf.shape[0]):
    row = X_tfidf[doc_idx].toarray().ravel()

    # índices dos termos com maior TF-IDF
    top_idx = row.argsort()[::-1][:TOP_N]

    print(f"\nDocumento {doc_idx}: {documents[doc_idx]}")

    for term_idx in top_idx:
        if row[term_idx] > 0:
            print(f"  TF-IDF {row[term_idx]:.4f} → termo '{vocab[term_idx]}'")



Documento 0: A NASA lançou um foguete para colocar um satélite em órbita e estudar a atmosfera.
  TF-IDF 0.3680 → termo 'atmosfera'
  TF-IDF 0.3680 → termo 'estudar'
  TF-IDF 0.3680 → termo 'lançou'
  TF-IDF 0.3680 → termo 'colocar'
  TF-IDF 0.3680 → termo 'nasa'
  TF-IDF 0.3281 → termo 'órbita'
  TF-IDF 0.3281 → termo 'foguete'
  TF-IDF 0.3281 → termo 'satélite'

Documento 1: Astrônomos analisaram dados de um telescópio espacial para observar galáxias distantes.
  TF-IDF 0.3662 → termo 'distantes'
  TF-IDF 0.3662 → termo 'dados'
  TF-IDF 0.3662 → termo 'astrônomos'
  TF-IDF 0.3662 → termo 'analisaram'
  TF-IDF 0.3662 → termo 'galáxias'
  TF-IDF 0.3662 → termo 'observar'
  TF-IDF 0.3265 → termo 'telescópio'
  TF-IDF 0.2983 → termo 'espacial'

Documento 2: A missão a Marte depende de propulsão eficiente, comunicação por rádio e planejamento orbital.
  TF-IDF 0.3229 → termo 'rádio'
  TF-IDF 0.3229 → termo 'comunicação'
  TF-IDF 0.3229 → termo 'eficiente'
  TF-IDF 0.3229 → termo 'marte'


## 4. Descoberta de estruturas latentes (Modelagem de Tópicos – NMF)

NMF (Non-negative Matrix Factorization - Fatoração de Matrizes Não Negativas) é uma técnica de fatoração matricial que decompõe uma matriz numérica com valores não negativos em duas outras matrizes (H e W), também não negativas, de menor dimensionalidade.

Formalmente, dado:
$$X \in \mathbb{R}_{\ge 0}^{n \times m}$$

o NMF busca encontrar:
$$W \in \mathbb{R}_{\ge 0}^{n \times k}, \quad H \in \mathbb{R}_{\ge 0}^{k \times m}$$

tais que: $X \approx W \cdot H$  
onde:  
- $X$ é a matriz original (`X_tfidf`)
- $W$ representa a associação entre objetos e componentes latentes  
- $H$ representa a associação entre componentes latentes e atributos  
- $k$ é o número de componentes (tópicos)


Aplicaremos NMF sobre a matriz `X_tfidf` para obter:
- `H[j, k]` no exemplo a seguir 0.3830 é quanto o termo 'órbita' define semanticamente o tópico 2.
```
Tópico 2: pesos dos termos mais representativos:
  H[2, 175] = 0.3830  → termo: 'órbita'
  H[2, 72] = 0.3830  → termo: 'foguete'
  H[2, 143] = 0.2890  → termo: 'satélite'
  H[2, 16] = 0.2290  → termo: 'atmosfera'
  H[2, 64] = 0.2290  → termo: 'estudar'
  H[2, 87] = 0.2290  → termo: 'lançou'
```
- `W[i, j]` no exemplo a seguir 0.7645 é quanto o documento 0 fala sobre o conteúdo semântico do tópico 2.
```
DOCUMENTO 0: A NASA lançou um foguete para colocar um satélite em órbita e estudar a atmosfera.  
W[0, 2] = 0.7645  
    → associação com TÓPICO 2  
    → conteúdo do tópico: órbita, foguete, satélite, atmosfera, estudar, lançou
```

O método `fit_transform` executa duas etapas inseparáveis:
1. fit → dada a matriz TF-IDF (`X_tfidf`), o NMF busca as matrizes `W` (documentos × tópicos) e `H` (tópicos × termos). Essa decomposição é aprendida sem rótulos, o que caracteriza descoberta de conhecimento;
2.  transform → mantém `H` fixo (tópicos aprendidos) e calcula `W` para os documentos.


> No NMF, `fit_transform` descobre os tópicos latentes do corpus e calcula **quanto cada documento pertence a cada tópico**, produzindo a matriz W, enquanto a matriz H descreve os tópicos em termos de palavras.

In [7]:
from sklearn.decomposition import NMF
import numpy as np

TOP_N_TERMS = 6
N_COMPONENTS = 3

# Ajuste do modelo
nmf = NMF(n_components=N_COMPONENTS, random_state=42)
W = nmf.fit_transform(X_tfidf)
H = nmf.components_
print(f"Dimensões: documents:{len(documents)}, X_tfidf:{X_tfidf.shape}, H: {H.shape}, W: {W.shape}")

print("\n=== MATRIZ H: Tópicos, termos e conteúdo semântico ===\n")

topic_contents = {}  # para reutilizar na matriz W

for j in range(H.shape[0]):  # para cada tópico
    # termos mais importantes do tópico j
    top_idx = np.argsort(H[j])[::-1][:TOP_N_TERMS]
    top_terms = [vocab[k] for k in top_idx]

    # guarda o conteúdo do tópico
    topic_contents[j] = top_terms

    print(f"Tópico {j}: pesos dos termos mais representativos:")
    for k in top_idx:
        print(f"  H[{j}, {k}] = {H[j, k]:.4f}  → termo: '{vocab[k]}'")

    print("\n" + "-" * 70 + "\n")

print("\n=== MATRIZ W: Associação dos documentos aos tópicos ===\n")

for i in range(W.shape[0]):  # para cada documento
    print(f"DOCUMENTO {i}: {documents[i]}\n")

    for j in range(W.shape[1]):  # para cada tópico
        print(f"  W[{i}, {j}] = {W[i, j]:.4f}")
        print(f"    → associação com TÓPICO {j}")
        print(f"    → conteúdo do tópico: {', '.join(topic_contents[j])}")

    # tópico dominante
    j_max = W[i].argmax()
    print("\nTópico dominante do documento:")
    print(f"  → Tópico {j_max} (W = {W[i, j_max]:.4f})")
    print(f"  → Conteúdo: {', '.join(topic_contents[j_max])}")

    print("\n" + "=" * 80 + "\n")


Dimensões: documents:30, X_tfidf:(30, 176), H: (3, 176), W: (30, 3)

=== MATRIZ H: Tópicos, termos e conteúdo semântico ===

Tópico 0: pesos dos termos mais representativos:
  H[0, 78] = 0.5548  → termo: 'gpu'
  H[0, 137] = 0.5031  → termo: 'renderização'
  H[0, 117] = 0.5020  → termo: 'pipeline'
  H[0, 148] = 0.4929  → termo: 'shaders'
  H[0, 160] = 0.3939  → termo: 'texturas'
  H[0, 79] = 0.3533  → termo: 'gráfico'

----------------------------------------------------------------------

Tópico 1: pesos dos termos mais representativos:
  H[1, 75] = 0.5307  → termo: 'gol'
  H[1, 91] = 0.4582  → termo: 'marcou'
  H[1, 21] = 0.3502  → termo: 'campo'
  H[1, 96] = 0.3502  → termo: 'meio'
  H[1, 13] = 0.3033  → termo: 'atacante'
  H[1, 113] = 0.2378  → termo: 'partida'

----------------------------------------------------------------------

Tópico 2: pesos dos termos mais representativos:
  H[2, 175] = 0.3830  → termo: 'órbita'
  H[2, 72] = 0.3830  → termo: 'foguete'
  H[2, 143] = 0.2890  →

## 5. Organização do corpus (Clustering – KMeans)

Nesta etapa do pipeline de KDT, aplicamos clustering para organizar automaticamente os documentos por similaridade, sem o uso de rótulos prévios. O objetivo não é identificar sobre o que os documentos falam, mas quais documentos são semelhantes entre si considerando sua representação vetorial.

O algoritmo utilizado é o K-Means, aplicado sobre a matriz TF-IDF (X_tfidf), na qual cada documento é representado como um ponto em um espaço vetorial de alta dimensão.

---

### O que o K-Means faz

O K-Means é um algoritmo de aprendizagem não supervisionada que:  
1.  Recebe como entrada:  
    - um conjunto de vetores (aqui, documentos representados por TF-IDF)  
    - um número fixo de grupos (*k*)  
2.  Inicializa *k* centróides no espaço vetorial  
3.  Atribui cada documento ao centróide mais próximo (segundo a distância euclidiana)  
4.  Recalcula os centróides como a média dos documentos atribuídos  
5.  Repete o processo até a convergência

O resultado final é:  
- uma partição do corpus em *k* clusters
- cada documento pertence exatamente a um cluster

No código abaixo, utilizamos `k = 3`, de forma coerente com o tamanho reduzido e o caráter didático do corpus.

---

### Diferença conceitual entre K-Means e NMF

- NMF (Non-negative Matrix Factorization)
  - Descobre tópicos latentes no corpus
  - Um documento pode estar associado a vários tópicos simultaneamente
  - Produz uma decomposição semântica (matrizes *W* e *H*)
- K-Means (Clustering)
  - Agrupa documentos por proximidade geométrica no espaço vetorial
  - Cada documento pertence a um único cluster
  - Produz uma organização do corpus, não uma decomposição temática  

Em termos simples:
> NMF responde à pergunta *“quais temas existem no corpus?”*,
enquanto K-Means responde à pergunta *“quais documentos são semelhantes entre si?”*

In [8]:
from sklearn.cluster import KMeans

# Ajuste do modelo de clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X_tfidf)

print("=== DOCUMENTOS AGRUPADOS POR CLUSTER ===")

for c in range(kmeans.n_clusters):
    print(f"\nCLUSTER {c}")
    print("=" * 30)

    for i in range(len(documents)):
        if clusters[i] == c:
            print(f"- Documento {i} ({labels[i]}): {documents[i]}")

=== DOCUMENTOS AGRUPADOS POR CLUSTER ===

CLUSTER 0
- Documento 10 (futebol): O time marcou gol após jogada pelo meio campo e venceu a partida.
- Documento 11 (futebol): O técnico ajustou o esquema tático e reforçou o meio campo.
- Documento 17 (futebol): A equipe dominou o meio campo e criou chances de gol.

CLUSTER 1
- Documento 9 (espaco): A estação espacial realizou experimentos de microgravidade com novos materiais.
- Documento 20 (computacao_grafica): A GPU acelera a renderização 3D com shaders e texturas no pipeline gráfico.
- Documento 21 (computacao_grafica): O pipeline de renderização usa GPU, shader e textura para desenhar a cena em tempo real.
- Documento 22 (computacao_grafica): Shaders controlam iluminação e materiais, enquanto a GPU aplica texturas na renderização.
- Documento 23 (computacao_grafica): A renderização em tempo real depende de GPU e do pipeline gráfico com shaders otimizados.
- Documento 24 (computacao_grafica): Ray tracing melhora sombras e reflexos, mas e

Na célula anterior observa-se que os documetnos relacionados a futebol e espaço foram agrupados como pertecentes ao cluster 2.Imprimir os termos característicos do centróide de cada cluster.

Isso acontecer por que o tema 2 misturou espaço e futebol, assim como pode ser visto a seguir.

In [9]:
import numpy as np

def print_cluster_centroid_terms(kmeans, cluster_id, vocab, top_n=10):
    centroid = kmeans.cluster_centers_[cluster_id]
    top_idx = centroid.argsort()[::-1][:top_n]
    print(f"\nCluster {cluster_id} – termos do centróide:")
    for k in top_idx:
        print(f"  peso {centroid[k]:.4f} → '{vocab[k]}'")

# Ex.: centróide do cluster 1 (computação)
for i in range(kmeans.n_clusters):
    print_cluster_centroid_terms(kmeans, i, vocab, top_n=12)


Cluster 0 – termos do centróide:
  peso 0.3184 → 'campo'
  peso 0.3184 → 'meio'
  peso 0.1809 → 'gol'
  peso 0.1381 → 'equipe'
  peso 0.1381 → 'criou'
  peso 0.1381 → 'chances'
  peso 0.1381 → 'dominou'
  peso 0.1347 → 'tático'
  peso 0.1347 → 'esquema'
  peso 0.1347 → 'técnico'
  peso 0.1347 → 'reforçou'
  peso 0.1201 → 'ajustou'

Cluster 1 – termos do centróide:
  peso 0.2020 → 'gpu'
  peso 0.1803 → 'renderização'
  peso 0.1682 → 'pipeline'
  peso 0.1580 → 'shaders'
  peso 0.1353 → 'texturas'
  peso 0.1001 → 'gráfico'
  peso 0.0847 → 'shader'
  peso 0.0657 → 'custo'
  peso 0.0656 → 'real'
  peso 0.0640 → 'materiais'
  peso 0.0630 → 'textura'
  peso 0.0599 → 'tempo'

Cluster 2 – termos do centróide:
  peso 0.0586 → 'marcou'
  peso 0.0549 → 'gol'
  peso 0.0471 → 'atacante'
  peso 0.0449 → 'gols'
  peso 0.0424 → 'defesa'
  peso 0.0418 → 'telescópio'
  peso 0.0398 → 'foguete'
  peso 0.0398 → 'órbita'
  peso 0.0397 → 'por'
  peso 0.0395 → 'satélite'
  peso 0.0370 → 'espacial'
  peso 0.02

### Clustering usando tópicos (matriz *W*)

Nesta etapa, o agrupamento dos documentos é realizado no espaço dos tópicos, utilizando a matriz `W` produzida pelo NMF, em vez da matriz TF-IDF (`X_tfidf`).

A matriz `W` representa cada documento como um vetor de graus de associação a tópicos latentes, isto é, cada linha indica quanto o documento aborda cada tópico descoberto previamente. Trata-se, portanto, de uma representação semântica reduzida, com dimensionalidade muito menor e menos ruído lexical do que a representação baseada diretamente em termos.

Ao aplicar o K-Means sobre W, o clustering passa a organizar o corpus com base em padrões temáticos globais, e não apenas na sobreposição de palavras. Com isso, documentos que compartilham distribuições de tópicos semelhantes tendem a ser agrupados, mesmo que utilizem vocabulários diferentes.

Com essa alteração, o clustering responde à seguinte pergunta *“Quais documentos apresentam perfis temáticos semelhantes?”*

Essa abordagem costuma produzir agrupamentos mais coerentes semanticamente, além de evidenciar a complementaridade entre:  
- NMF, que descobre e modela tópicos latentes, e
- K-Means, que organiza documentos a partir dessas representações temáticas.

In [10]:
from sklearn.cluster import KMeans

# Ajuste do modelo de clustering usando a matriz W (documentos × tópicos)
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(W)

print("=== DOCUMENTOS AGRUPADOS POR CLUSTER (USANDO A MATRIZ W) ===")

for c in range(kmeans.n_clusters):
    print(f"\nCLUSTER {c}")
    print("=" * 30)

    for i in range(len(documents)):
        if clusters[i] == c:
            print(f"- Documento {i} ({labels[i]}): {documents[i]}")

=== DOCUMENTOS AGRUPADOS POR CLUSTER (USANDO A MATRIZ W) ===

CLUSTER 0
- Documento 1 (espaco): Astrônomos analisaram dados de um telescópio espacial para observar galáxias distantes.
- Documento 2 (espaco): A missão a Marte depende de propulsão eficiente, comunicação por rádio e planejamento orbital.
- Documento 4 (espaco): A agência espacial discutiu janelas de lançamento, trajetória e reentrada controlada.
- Documento 5 (espaco): Cientistas monitoraram a atividade solar e seus efeitos em comunicações e navegação.
- Documento 6 (espaco): O rover coletou amostras de solo marciano e transmitiu resultados para a base.
- Documento 7 (espaco): O telescópio detectou exoplanetas pela variação de brilho e espectroscopia.
- Documento 9 (espaco): A estação espacial realizou experimentos de microgravidade com novos materiais.
- Documento 10 (futebol): O time marcou gol após jogada pelo meio campo e venceu a partida.
- Documento 11 (futebol): O técnico ajustou o esquema tático e reforçou o meio 

---
## 6. Estruturação de conhecimento (tabelas + mini-grafo de coocorrências)

Nesta etapa do pipeline de KDT, transformamos os resultados numéricos obtidos nas fases anteriores em estruturas explícitas de conhecimento, que podem ser inspecionadas, armazenadas ou utilizadas por outros sistemas.

Diferentemente das etapas anteriores, que produzem representações internas (vetores, matrizes e pesos), o objetivo aqui é gerar saídas estruturadas e interpretáveis, típicas de sistemas de KDT.

Nesta célula, produzimos duas estruturas principais:
1. Tabela lógica documento → atributos, contendo:
    - categoria original (apenas para validação didática),  
    - tópico dominante (obtido a partir da matriz `W`),  
    - cluster atribuído (obtido pelo K-Means).
2.  Lista de arestas (edge list) representando um mini-grafo de coocorrência de termos, no qual:
    - cada nó corresponde a um termo relevante,  
    - cada aresta indica que dois termos coocorrem entre os termos mais significativos de um mesmo documento,  
    - o peso da aresta indica a frequência dessa coocorrência no corpus.

Nesse momento, o texto deixa de ser apenas texto e passa a ser representado como **conhecimento estruturado**, caracterizando efetivamente o processo de KDT.

In [11]:
print("\n=== TABELA LÓGICA: DOCUMENTO → ATRIBUTOS ===\n")

for i in range(len(documents)):
    topico_dominante = W[i].argmax()
    print(f"Documento {i}: {documents[i]}")
    print(f"Categoria original: {labels[i]}")
    print(f"Tópico dominante: {topico_dominante} (W = {W[i, topico_dominante]:.4f})")
    print(f"Cluster atribuído: {clusters[i]}")
    print("-" * 80)



=== TABELA LÓGICA: DOCUMENTO → ATRIBUTOS ===

Documento 0: A NASA lançou um foguete para colocar um satélite em órbita e estudar a atmosfera.
Categoria original: espaco
Tópico dominante: 2 (W = 0.7645)
Cluster atribuído: 2
--------------------------------------------------------------------------------
Documento 1: Astrônomos analisaram dados de um telescópio espacial para observar galáxias distantes.
Categoria original: espaco
Tópico dominante: 0 (W = 0.0012)
Cluster atribuído: 0
--------------------------------------------------------------------------------
Documento 2: A missão a Marte depende de propulsão eficiente, comunicação por rádio e planejamento orbital.
Categoria original: espaco
Tópico dominante: 0 (W = 0.0294)
Cluster atribuído: 0
--------------------------------------------------------------------------------
Documento 3: O satélite enviou imagens da Terra e medições de radiação no cinturão de Van Allen.
Categoria original: espaco
Tópico dominante: 2 (W = 0.3423)
Clust

### Como interpretar a "Edge List" (Lista de Arestas)

A tabela gerada acima representa um **grafo simplificado**:
- **Nós (Nodes):** representam os termos (palavras-chave).
- **Arestas (Edges):** conectam termos que aparecem juntos entre os principais termos (Top TF-IDF) de um mesmo documento.
- **Peso (Weight):** indica a frequência com que essa coocorrência foi observada no corpus.

Em aplicações avançadas de KDT, esse processo de estruturação evolui para:
- **Coocorrência de Entidades:** conexões baseadas em entidades nomeadas (via NER).
- **Extração de Relações:** identificação de triplas semânticas (Sujeito–Verbo–Objeto).
- **Construção de Grafos de Conhecimento:** mapeamento formal de conexões do tipo (Entidade–Relação–Entidade).

In [12]:
print("\n=== MINI-GRAFO DE COOCCORRÊNCIA (EDGE LIST) ===\n")

from collections import Counter

pair_counter = Counter()
TOP_TERMS = 4  # número de termos mais significativos por documento

for i in range(X_tfidf.shape[0]):
    row = X_tfidf[i].toarray().ravel()

    # índices dos termos mais relevantes do documento i
    idx = row.argsort()[::-1][:TOP_TERMS]
    terms = [vocab[j] for j in idx if row[j] > 0]

    # gerar pares de coocorrência
    for a in range(len(terms)):
        for b in range(a + 1, len(terms)):
            pair_counter[(terms[a], terms[b])] += 1


for (termo_1, termo_2), peso in pair_counter.most_common():
    print(f"Aresta ({termo_1}, {termo_2}) → peso = {peso}")



=== MINI-GRAFO DE COOCCORRÊNCIA (EDGE LIST) ===

Aresta (atmosfera, estudar) → peso = 1
Aresta (atmosfera, lançou) → peso = 1
Aresta (atmosfera, colocar) → peso = 1
Aresta (estudar, lançou) → peso = 1
Aresta (estudar, colocar) → peso = 1
Aresta (lançou, colocar) → peso = 1
Aresta (distantes, dados) → peso = 1
Aresta (distantes, astrônomos) → peso = 1
Aresta (distantes, analisaram) → peso = 1
Aresta (dados, astrônomos) → peso = 1
Aresta (dados, analisaram) → peso = 1
Aresta (astrônomos, analisaram) → peso = 1
Aresta (rádio, comunicação) → peso = 1
Aresta (rádio, eficiente) → peso = 1
Aresta (rádio, marte) → peso = 1
Aresta (comunicação, eficiente) → peso = 1
Aresta (comunicação, marte) → peso = 1
Aresta (eficiente, marte) → peso = 1
Aresta (van, terra) → peso = 1
Aresta (van, medições) → peso = 1
Aresta (van, allen) → peso = 1
Aresta (terra, medições) → peso = 1
Aresta (terra, allen) → peso = 1
Aresta (medições, allen) → peso = 1
Aresta (trajetória, reentrada) → peso = 1
Aresta (trajet

## 7. Fechamento: O Pipeline de KDT (O que você deve fixar)

### 1️⃣ O que não é KDT (Pré-requisitos)
As etapas iniciais preparam o terreno, mas ainda não constituem "descoberta" de conhecimento novo, apenas formatação.

* **🔹 Pré-processamento (`documents_clean`)**
    * **O que é:** Limpeza, remoção de ruído e normalização.
    * **Papel:** Preparação técnica. Sem isso, o algoritmo "enxerga" sujeira, não padrões.
* **🔹 Representação Vetorial (TF-IDF)**
    * **O que é:** Transformação de palavras em pesos numéricos.
    ```python
    X_tfidf = vectorizer.fit_transform(documents_clean)
    ```
    * **Papel:** Infraestrutura matemática. É a "ponte" entre o texto humano e o cálculo computacional.

---

### 2️⃣ Onde o KDT começa de fato
A descoberta surge quando o algoritmo identifica padrões que não foram explicitamente rotulados por humanos.

* **🔹 Modelagem de Tópicos (NMF)**
    ```python
    W = nmf.fit_transform(X_tfidf)
    H = nmf.components_
    ```
    * **A Descoberta:** O algoritmo revela **estruturas latentes** (temas ocultos).
    * **O Insight:** Surgem conceitos semânticos globais (ex: "Tópico de Astronomia" vs. "Tópico de Esportes") baseados apenas na coocorrência estatística. É o KDT em sua forma **exploratória**.

---

### 3️⃣ KDT em nível intermediário: Organização do Corpus
* **🔹 Clustering (K-Means sobre a matriz W)**
    ```python
    clusters = kmeans.fit_predict(W)
    ```
    * **A Descoberta:** O corpus deixa de ser uma lista desordenada e passa a ser uma **estrutura organizada**.
    * **O Insight:** Documentos são agrupados por similaridade temática, permitindo mapear a taxonomia (as divisões) do seu conjunto de dados automaticamente.

---

### 4️⃣ Onde o KDT se consolida: Estruturação de Conhecimento
Este é o objetivo final: **transformar dados não estruturados em dados estruturados (tabelas/grafos).**

* **🔹 Tabela Lógica de Atributos**
    O núcleo do KDT neste notebook é a consolidação de informações dispersas em uma estrutura relacional clara:
```
Documento i →
    categoria original
    tópico dominante
    cluster
```
* **🔹 Lista de Arestas (Edge List)**
    Extração de conexões entre termos para visualização de redes ou alimentação de **Grafos de Conhecimento**.

> **Conclusão:** O KDT não é um algoritmo único, mas o **processo** de transformar grandes volumes de texto em informações acionáveis e organizadas.

## Exercício - Novo Corpus para KDT

Neste exercício, você deverá substituir o corpus original do notebook pelo corpus fornecido a seguir e adaptar o pipeline de KDT para lidar com um corpus composto por quatro temas distintos.

As adaptações devem incluir, entre outros ajustes necessários:
- a escolha adequada do número de componentes do NMF (`N_COMPONENTS`);
- a análise do impacto dessas mudanças nos tópicos, clusters e estruturas de conhecimento geradas.

O objetivo do exercício é compreender que KDT é um processo exploratório, no qual:
- os tópicos descobertos dependem dos dados,
- o número de tópicos não corresponde automaticamente ao número de temas esperados,
- e as estruturas finais de conhecimento são sensíveis às escolhas de modelagem feitas ao longo do pipeline.

In [2]:
politica_docs = [
    "O governo apresentou uma proposta de reforma administrativa ao congresso.",
    "O presidente discursou sobre democracia, instituições e estabilidade política.",
    "Deputados debateram projetos de lei relacionados à transparência pública.",
    "A campanha eleitoral focou em segurança, educação e políticas sociais.",
    "O partido articulou alianças para garantir maioria no parlamento.",
    "O senador criticou a condução da política externa do governo.",
    "O congresso aprovou medidas provisórias após intensas negociações.",
    "A oposição questionou a legitimidade das decisões do executivo.",
    "O debate político foi marcado por polarização ideológica.",
    "Reformas institucionais dominaram a agenda política da semana."
]

economia_docs = [
    "A inflação acumulada impactou o poder de compra da população.",
    "O banco central ajustou a taxa de juros para conter a inflação.",
    "Indicadores econômicos apontam crescimento do produto interno bruto.",
    "O mercado financeiro reagiu às decisões de política monetária.",
    "A alta do dólar afetou exportações e importações.",
    "Investidores analisaram riscos fiscais e expectativas de crescimento.",
    "O desemprego apresentou queda segundo dados oficiais.",
    "O governo anunciou medidas para estimular a atividade econômica.",
    "A política econômica busca equilíbrio entre crescimento e controle inflacionário.",
    "Relatórios econômicos destacaram a desaceleração global."
]

saude_docs = [
    "Pesquisadores desenvolveram uma nova vacina contra doenças respiratórias.",
    "O sistema de saúde enfrentou sobrecarga durante o surto epidemiológico.",
    "Estudos clínicos avaliaram a eficácia de novos medicamentos.",
    "A prevenção é fundamental para reduzir riscos à saúde pública.",
    "Hospitais adotaram protocolos para melhorar o atendimento aos pacientes.",
    "A saúde mental ganhou destaque em políticas públicas recentes.",
    "Campanhas de vacinação ampliaram a cobertura da população.",
    "Profissionais de saúde alertaram para a importância do diagnóstico precoce.",
    "A pesquisa médica avançou com novos tratamentos.",
    "Indicadores de saúde mostram melhora na expectativa de vida."
]

ti_docs = [
    "Sistemas de informação utilizam bancos de dados para armazenar dados.",
    "A computação em nuvem permite escalabilidade e redução de custos.",
    "Algoritmos de aprendizado de máquina analisam grandes volumes de dados.",
    "A segurança da informação é essencial para proteger sistemas.",
    "Arquiteturas de software modernas utilizam microsserviços.",
    "O processamento distribuído melhora o desempenho de aplicações.",
    "Redes de computadores permitem comunicação entre dispositivos.",
    "A inteligência artificial impulsiona novas soluções tecnológicas.",
    "O desenvolvimento de software exige boas práticas de engenharia.",
    "Soluções digitais transformaram processos organizacionais."
]

labels = (
    ["politica"] * len(politica_docs) +
    ["economia"] * len(economia_docs) +
    ["saude"] * len(saude_docs) +
    ["tecnologia"] * len(ti_docs)
)

documents = politica_docs + economia_docs + saude_docs + ti_docs

print("Total de documentos:", len(documents))
print("Exemplo:", documents[0]) # mostra apenas o 1o documento


Total de documentos: 40
Exemplo: O governo apresentou uma proposta de reforma administrativa ao congresso.
